В связи с высокой долей пропусков и нестабильностью адресных полей сопоставление объектов источников A и B по адресам считается ненадёжным.
Поэтому основной принцип слияния — пространственное сопоставление геометрий.

---

При этом в задаче возможны не только простые случаи один-к-одному , но и:
- Один-ко-многим
- Многие-к-одному
- Многие-ко-многим

Поскольку одно и то же физическое здание может быть представлено несколькими полигонами в одном или обоих источниках.

---

По этой причине слияние реализуется не как жёсткое соединение по лучшей паре, а как граф пространственных связей:
1. между объектами A и B строятся кандидатные пары по spatial index
2. для каждой пары вычисляются метрики перекрытия и близости
3. на основании пороговых правил формируются рёбра графа
4. компоненты связности графа интерпретируются как кандидаты на единое физическое здание
5. для каждой компоненты выбирается репрезентативная геометрия и агрегируются доступные атрибуты источника B

## Этап 1

---

На этом этапе мы:
1. загружаем уже очищенные датасеты A и B из parquet
2. оставляем только релевантные здания
3. приводим геометрию к единой метрической системе координат
4. создаём технические идентификаторы объектов
5. считаем базовые геометрические характеристики, которые понадобятся для сопоставления и последующей агрегации.

In [10]:
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from pathlib import Path

In [11]:
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

In [12]:
DATA_DIR = Path("../data/interim")
OUTPUT_DIR = Path("../data/interim")
SOURCE_A_PATH = DATA_DIR / "A_clean_final1.parquet"
SOURCE_B_PATH = DATA_DIR / "B_clean_final1.parquet"

In [13]:
TARGET_CRS = "EPSG:32635"

gdf_a = gpd.read_parquet(SOURCE_A_PATH)
gdf_b = gpd.read_parquet(SOURCE_B_PATH)

In [14]:
print("Исходные размеры после загрузки parquet:")
print(f"A: {len(gdf_a):,}")
print(f"B: {len(gdf_b):,}")

Исходные размеры после загрузки parquet:
A: 169,517
B: 160,975


In [15]:
# ---------- фильтр по релевантным зданиям ----------
if "is_relevant_building" in gdf_a.columns:
    gdf_a = gdf_a[gdf_a["is_relevant_building"]].copy()

if "is_relevant_building" in gdf_b.columns:
    gdf_b = gdf_b[gdf_b["is_relevant_building"]].copy()

print("\nПосле фильтра is_relevant_building:")
print(f"A: {len(gdf_a):,}")
print(f"B: {len(gdf_b):,}")


После фильтра is_relevant_building:
A: 169,517
B: 160,947


In [16]:
# ---------- перевод в метрическую CRS ----------
gdf_a = gdf_a.to_crs(TARGET_CRS)
gdf_b = gdf_b.to_crs(TARGET_CRS)

In [17]:
# ---------- служебные идентификаторы ----------
gdf_a = gdf_a.reset_index(drop=False).rename(columns={"index": "orig_index_a"})
gdf_b = gdf_b.reset_index(drop=False).rename(columns={"index": "orig_index_b"})

gdf_a["src"] = "A"
gdf_b["src"] = "B"

gdf_a["uid"] = "A_" + gdf_a.index.astype(str)
gdf_b["uid"] = "B_" + gdf_b.index.astype(str)

In [18]:
# ---------- базовые геометрические признаки ----------
gdf_a["area_m2"] = gdf_a.geometry.area
gdf_b["area_m2"] = gdf_b.geometry.area

gdf_a["perimeter_m"] = gdf_a.geometry.length
gdf_b["perimeter_m"] = gdf_b.geometry.length

gdf_a["centroid"] = gdf_a.geometry.centroid
gdf_b["centroid"] = gdf_b.geometry.centroid

print("\nБазовая статистика площадей:")
print(gdf_a["area_m2"].describe(percentiles=[0.25, 0.5, 0.75]))
print()
print(gdf_b["area_m2"].describe(percentiles=[0.25, 0.5, 0.75]))


Базовая статистика площадей:
count    169517.000000
mean        509.913779
std        1472.014256
min           4.032179
25%          41.527680
50%         122.311725
75%         525.524630
max      137722.730007
Name: area_m2, dtype: float64

count    160947.000000
mean        489.338696
std        1301.527683
min           3.774270
25%          42.994492
50%         125.102143
75%         534.423109
max       72648.586353
Name: area_m2, dtype: float64


## Этап 2

---

В задаче нельзя ограничиваться жёстким соединением, как было обговорено ранее, поэтому:
1. сначала найдём кандидатов на связь A-B по пространственному индексу
2. затем посчитаем метрики перекрытия
3. затем по правилам сформируем рёбра графа
4. после этого выделим компоненты связности

---

Для ускорения расчётов заранее создаём словари геометрий и площадей по uid, чтобы быстро обращаться к объектам при вычислении попарных метрик

In [19]:
# Словари для быстрого доступа
a_geom = gdf_a.set_index("uid").geometry.to_dict()
b_geom = gdf_b.set_index("uid").geometry.to_dict()

a_area = gdf_a.set_index("uid")["area_m2"].to_dict()
b_area = gdf_b.set_index("uid")["area_m2"].to_dict()

# Для удобства оставим укороченные таблицы
a_small = gdf_a[["uid", "geometry", "area_m2", "perimeter_m"]].copy()
b_small = gdf_b[["uid", "geometry", "area_m2", "perimeter_m"]].copy()

print("Технические таблицы подготовлены.")
print(f"a_small: {a_small.shape}")
print(f"b_small: {b_small.shape}")

Технические таблицы подготовлены.
a_small: (169517, 4)
b_small: (160947, 4)


## Шаг 3 - Поиск кандидатных пар A-B через spatial index

---

На этом шаге мы строим только кандидатов на сопоставление, а не финальные матчи.

Почему используем буфер:
- границы одного и того же здания в разных источниках могут быть сдвинуты
- может быть микрорасхождение контуров

Поэтому мы не ищем только строгое пересечение "polygon intersects polygon", а слегка расширяем объекты A на небольшой допуск и ищем пересечения с B.

Здесь buffer_tol_m = 0.5 метра — это аккуратный стартовый порог.

In [20]:
buffer_tol_m = 0.5

In [21]:
a_buffered = a_small.copy()
a_buffered["geometry"] = a_buffered.geometry.buffer(buffer_tol_m)

candidate_pairs = gpd.sjoin(
    a_buffered,
    b_small,
    how="inner",
    predicate="intersects"
).reset_index(drop=True)

# Переименуем колонки в более удобный вид
candidate_pairs = candidate_pairs.rename(columns={
    "uid_left": "uid_a",
    "uid_right": "uid_b",
    "area_m2_left": "area_a",
    "area_m2_right": "area_b",
    "perimeter_m_left": "perimeter_a",
    "perimeter_m_right": "perimeter_b"
})

print("Количество кандидатных пар A-B после spatial join:")
print(f"{len(candidate_pairs):,}")

candidate_pairs.head()

Количество кандидатных пар A-B после spatial join:
242,604


,uid_a,geometry,area_a,perimeter_a,index_right,uid_b,area_b,perimeter_b
0,A_0,"POLYGON ((673846.903 6635483.248, 673846.887 6...",84.998796,59.420137,5345,B_5345,1452.385070,291.455622
1,A_1,"MULTIPOLYGON (((673877.396 6635487.484, 673877...",114.686068,77.274278,5585,B_5585,1397.525977,267.926214
2,A_2,"POLYGON ((677035.684 6640339.288, 677035.637 6...",122.970600,236.348518,24877,B_24877,2624.109343,628.415722
3,A_5,"POLYGON ((678587.398 6638710.712, 678587.367 6...",83.684805,63.645808,29605,B_29605,549.544764,158.404694
4,A_6,"POLYGON ((678902.057 6640087.874, 678902.037 6...",42.763701,135.255356,30354,B_30354,554.767129,129.273252


## Этап 4 - Расчёт попарных геометрических метрик для кандидатных пар

---

На этом этапе для каждой пары A-B считаем набор метрик, которые помогают понять, действительно ли два полигона относятся к одному физическому зданию

Мы считаем:
- intersection_area: площадь пересечения;
- union_area: площадь объединения;
- iou: классический Intersection over Union;
- overlap_a: доля покрытия объекта A объектом B;
- overlap_b: доля покрытия объекта B объектом A;
- dist_m: расстояние между геометриями;
- centroid_dist_m: расстояние между центроидами.

---
Идея:
- IoU — симметричная мера похожести
- overlap_a / overlap_b — асимметричные меры вложенности/покрытия

In [22]:
def calc_pair_metrics(uid_a: str, uid_b: str) -> dict:
    ga = a_geom[uid_a]
    gb = b_geom[uid_b]

    inter_geom = ga.intersection(gb)
    inter_area = inter_geom.area

    union_area = ga.union(gb).area
    iou = inter_area / union_area if union_area > 0 else 0.0

    area_a = a_area[uid_a]
    area_b = b_area[uid_b]

    overlap_a = inter_area / area_a if area_a > 0 else 0.0
    overlap_b = inter_area / area_b if area_b > 0 else 0.0

    dist_m = ga.distance(gb)
    centroid_dist_m = ga.centroid.distance(gb.centroid)

    return {
        "intersection_area": inter_area,
        "union_area": union_area,
        "iou": iou,
        "overlap_a": overlap_a,
        "overlap_b": overlap_b,
        "dist_m": dist_m,
        "centroid_dist_m": centroid_dist_m,
    }

In [23]:
metrics = candidate_pairs[["uid_a", "uid_b"]].apply(
    lambda row: pd.Series(calc_pair_metrics(row["uid_a"], row["uid_b"])),
    axis=1
)

candidate_pairs = pd.concat([candidate_pairs, metrics], axis=1)

print("Метрики рассчитаны.")
print(candidate_pairs[["iou", "overlap_a", "overlap_b", "dist_m"]].describe())

candidate_pairs.head()

Метрики рассчитаны.
                 iou      overlap_a      overlap_b         dist_m
count  242604.000000  242604.000000  242604.000000  242604.000000
mean        0.227642       0.382346       0.443244       0.009803
std         0.258385       0.335625       0.371324       0.056242
min         0.000000       0.000000       0.000000       0.000000
25%         0.013754       0.037405       0.042026       0.000000
50%         0.096277       0.324674       0.426785       0.000000
75%         0.418774       0.698591       0.803125       0.000000
max         0.980935       1.000000       1.000000       0.499991


,uid_a,geometry,area_a,perimeter_a,index_right,uid_b,area_b,perimeter_b,intersection_area,union_area,iou,overlap_a,overlap_b,dist_m,centroid_dist_m
0,A_0,"POLYGON ((673846.903 6635483.248, 673846.887 6...",84.998796,59.420137,5345,B_5345,1452.385070,291.455622,81.282333,1456.101533,0.055822,0.956276,0.055965,0.0,14.282694
1,A_1,"MULTIPOLYGON (((673877.396 6635487.484, 673877...",114.686068,77.274278,5585,B_5585,1397.525977,267.926214,113.291309,1398.920736,0.080985,0.987838,0.081066,0.0,17.169572
2,A_2,"POLYGON ((677035.684 6640339.288, 677035.637 6...",122.970600,236.348518,24877,B_24877,2624.109343,628.415722,15.619691,2731.460253,0.005718,0.127020,0.005952,0.0,44.066118
3,A_5,"POLYGON ((678587.398 6638710.712, 678587.367 6...",83.684805,63.645808,29605,B_29605,549.544764,158.404694,76.680053,556.549517,0.137778,0.916296,0.139534,0.0,11.871475
4,A_6,"POLYGON ((678902.057 6640087.874, 678902.037 6...",42.763701,135.255356,30354,B_30354,554.767129,129.273252,36.546872,560.983957,0.065148,0.854624,0.065878,0.0,1.220141


## Шаг 5 - Формирование рёбер графа по правилам пространственной связи

---

Теперь из кандидатных пар нужно выбрать только те, которые действительно считаем пространственно связанными.

Здесь используется не одно правило, а несколько альтернативных:
1. IoU >= 0.15 - Ловит случаи, когда контуры в целом похожи и достаточно пересекаются.
2. overlap_a >= 0.60 - Ловит случаи, когда почти весь объект A покрыт объектом B. Это важно для сценария "малый контур A внутри более общего контура B".
3. overlap_b >= 0.60 - Аналогично ловит обратный сценарий.
4. distance <= 1 м и есть хотя бы небольшое реальное пересечение относительно меньшего объекта. Это страховка для случаев очень близких контуров и микрорасхождений.

Такие правила дают:
- нормальную чувствительность к one-to-one;
- устойчивость к секционированию;
- меньшую вероятность недосопоставления.

In [24]:
IOU_THR = 0.15
OVERLAP_THR = 0.60
DIST_THR_M = 1.0
MIN_REL_INTERSECTION = 0.05

In [25]:
candidate_pairs["min_area"] = candidate_pairs[["area_a", "area_b"]].min(axis=1)
candidate_pairs["rel_intersection_min"] = np.where(
    candidate_pairs["min_area"] > 0,
    candidate_pairs["intersection_area"] / candidate_pairs["min_area"],
    0.0
)

In [26]:
edge_mask = (
    (candidate_pairs["iou"] >= IOU_THR) |
    (candidate_pairs["overlap_a"] >= OVERLAP_THR) |
    (candidate_pairs["overlap_b"] >= OVERLAP_THR) |
    (
        (candidate_pairs["dist_m"] <= DIST_THR_M) &
        (candidate_pairs["rel_intersection_min"] >= MIN_REL_INTERSECTION)
    )
)

In [27]:
edges_ab = candidate_pairs[edge_mask].copy()

print("После фильтра пространственных связей осталось рёбер:")
print(f"{len(edges_ab):,}")

После фильтра пространственных связей осталось рёбер:
208,798


In [28]:
print("\nРаспределение по основным условиям:")
print(f"IoU >= {IOU_THR}: {(candidate_pairs['iou'] >= IOU_THR).sum():,}")
print(f"overlap_a >= {OVERLAP_THR}: {(candidate_pairs['overlap_a'] >= OVERLAP_THR).sum():,}")
print(f"overlap_b >= {OVERLAP_THR}: {(candidate_pairs['overlap_b'] >= OVERLAP_THR).sum():,}")
print(f"distance <= {DIST_THR_M} м и rel_intersection >= {MIN_REL_INTERSECTION}: {(((candidate_pairs['dist_m'] <= DIST_THR_M) & (candidate_pairs['rel_intersection_min'] >= MIN_REL_INTERSECTION))).sum():,}")



Распределение по основным условиям:
IoU >= 0.15: 107,605
overlap_a >= 0.6: 80,009
overlap_b >= 0.6: 98,801
distance <= 1.0 м и rel_intersection >= 0.05: 208,798


In [29]:
edges_ab.head()

,uid_a,geometry,area_a,perimeter_a,index_right,uid_b,area_b,perimeter_b,intersection_area,union_area,iou,overlap_a,overlap_b,dist_m,centroid_dist_m,min_area,rel_intersection_min
0,A_0,"POLYGON ((673846.903 6635483.248, 673846.887 6...",84.998796,59.420137,5345,B_5345,1452.385070,291.455622,81.282333,1456.101533,0.055822,0.956276,0.055965,0.0,14.282694,84.998796,0.956276
1,A_1,"MULTIPOLYGON (((673877.396 6635487.484, 673877...",114.686068,77.274278,5585,B_5585,1397.525977,267.926214,113.291309,1398.920736,0.080985,0.987838,0.081066,0.0,17.169572,114.686068,0.987838
2,A_2,"POLYGON ((677035.684 6640339.288, 677035.637 6...",122.970600,236.348518,24877,B_24877,2624.109343,628.415722,15.619691,2731.460253,0.005718,0.127020,0.005952,0.0,44.066118,122.970600,0.127020
3,A_5,"POLYGON ((678587.398 6638710.712, 678587.367 6...",83.684805,63.645808,29605,B_29605,549.544764,158.404694,76.680053,556.549517,0.137778,0.916296,0.139534,0.0,11.871475,83.684805,0.916296
4,A_6,"POLYGON ((678902.057 6640087.874, 678902.037 6...",42.763701,135.255356,30354,B_30354,554.767129,129.273252,36.546872,560.983957,0.065148,0.854624,0.065878,0.0,1.220141,42.763701,0.854624


## 6 - Построение графа связей и выделение компонент связности

---

 Теперь переводим межисточниковые связи в граф:
- вершины = отдельные объекты A и B;
- ребро = достаточное пространственное соответствие между объектом A и B.

In [30]:
G = nx.Graph()

# Добавляем вершины
for uid in gdf_a["uid"]:
    G.add_node(uid, src="A")

for uid in gdf_b["uid"]:
    G.add_node(uid, src="B")

# Добавляем рёбра
for _, row in edges_ab.iterrows():
    G.add_edge(
        row["uid_a"],
        row["uid_b"],
        iou=row["iou"],
        overlap_a=row["overlap_a"],
        overlap_b=row["overlap_b"],
        dist_m=row["dist_m"],
        intersection_area=row["intersection_area"],
    )

components = list(nx.connected_components(G))

print("Граф построен.")
print(f"Количество компонент связности: {len(components):,}")

Граф построен.
Количество компонент связности: 139,649


In [31]:
component_sizes = pd.Series([len(c) for c in components])
print("\nРаспределение размеров компонент:")
print(component_sizes.value_counts().sort_index().head(20))


Распределение размеров компонент:
1     66070
2     49170
3      7852
4      5102
5      2122
6      2124
7      1165
8      1063
9       727
10      700
11      425
12      443
13      264
14      359
15      203
16      302
17      135
18      166
19      144
20      136
Name: count, dtype: int64


## Шаг 7 - Разбор компонент: состав, тип матча, базовые агрегаты

---

На этом этапе превращаем компоненты графа в табличный вид.

Для каждой компоненты считаем:
- список uid из A;
- список uid из B;
- число объектов из A и B;
- тип матча:

       A_only, B_only, 1:1, 1:n, n:1, n:n
 - максимальные и средние метрики по рёбрам внутри компоненты.

In [32]:
edge_lookup = edges_ab.set_index(["uid_a", "uid_b"])[
    ["iou", "overlap_a", "overlap_b", "dist_m", "intersection_area"]
].to_dict("index")

In [33]:
def get_component_edge_metrics(nodes):
    nodes = list(nodes)
    a_nodes = [n for n in nodes if n.startswith("A_")]
    b_nodes = [n for n in nodes if n.startswith("B_")]

    rows = []
    for a in a_nodes:
        for b in b_nodes:
            key = (a, b)
            if key in edge_lookup:
                row = edge_lookup[key].copy()
                row["uid_a"] = a
                row["uid_b"] = b
                rows.append(row)

    if not rows:
        return {
            "n_edges_ab": 0,
            "max_iou": np.nan,
            "mean_iou": np.nan,
            "max_overlap_a": np.nan,
            "max_overlap_b": np.nan,
            "min_dist_m": np.nan,
        }

    tmp = pd.DataFrame(rows)
    return {
        "n_edges_ab": len(tmp),
        "max_iou": tmp["iou"].max(),
        "mean_iou": tmp["iou"].mean(),
        "max_overlap_a": tmp["overlap_a"].max(),
        "max_overlap_b": tmp["overlap_b"].max(),
        "min_dist_m": tmp["dist_m"].min(),
    }

In [34]:
def classify_match_type(n_a, n_b):
    if n_a > 0 and n_b == 0:
        return "A_only"
    if n_b > 0 and n_a == 0:
        return "B_only"
    if n_a == 1 and n_b == 1:
        return "1:1"
    if n_a == 1 and n_b > 1:
        return "1:n"
    if n_a > 1 and n_b == 1:
        return "n:1"
    if n_a > 1 and n_b > 1:
        return "n:n"
    return "unknown"

In [35]:
component_rows = []

for comp_id, comp_nodes in enumerate(components, start=1):
    comp_nodes = sorted(comp_nodes)
    a_nodes = [n for n in comp_nodes if n.startswith("A_")]
    b_nodes = [n for n in comp_nodes if n.startswith("B_")]

    metrics_dict = get_component_edge_metrics(comp_nodes)

    component_rows.append({
        "component_id": comp_id,
        "uids_a": a_nodes,
        "uids_b": b_nodes,
        "n_a": len(a_nodes),
        "n_b": len(b_nodes),
        "match_type": classify_match_type(len(a_nodes), len(b_nodes)),
        **metrics_dict
    })


In [36]:
components_df = pd.DataFrame(component_rows)

In [37]:
print("Таблица компонент построена.")
print(components_df["match_type"].value_counts(dropna=False))

Таблица компонент построена.
match_type
1:1       49170
A_only    38086
B_only    27984
n:n       12201
1:n        6347
n:1        5861
Name: count, dtype: int64


In [38]:
components_df.head()

,component_id,uids_a,uids_b,n_a,n_b,match_type,n_edges_ab,max_iou,mean_iou,max_overlap_a,max_overlap_b,min_dist_m
0,1,"[A_0, A_3927, A_4013, A_4044, A_4134, A_4167, ...",[B_5345],8,1,n:1,8,0.099540,0.058140,0.965276,0.099897,0.0
1,2,"[A_1, A_4185, A_4263, A_4352, A_4414]",[B_5585],5,1,n:1,5,0.150684,0.093211,0.987838,0.151449,0.0
2,3,"[A_19400, A_19424, A_19431, A_19470, A_19514, ...","[B_24877, B_24899, B_24937, B_25028, B_25044, ...",12,8,n:n,27,0.489267,0.096725,1.000000,1.000000,0.0
3,4,[A_3],[],1,0,A_only,0,NaN,NaN,NaN,NaN,NaN
4,5,[A_4],[],1,0,A_only,0,NaN,NaN,NaN,NaN,NaN


## Шаг 8 - Добавление агрегированных геометрических характеристик компонентов

---

Для дальнейшей работы полезно сразу посчитать размеры и пространственные характеристики не только отдельных объектов, но и целых компонент.

Для каждой компоненты считаем:
- суммарную площадь объектов A;
- суммарную площадь объектов B;
- площадь объединения геометрий A внутри компоненты;
- площадь объединения геометрий B внутри компоненты;
- общую площадь объединённой геометрии всех объектов компоненты.

In [39]:
a_geom_by_uid = gdf_a.set_index("uid").geometry.to_dict()
b_geom_by_uid = gdf_b.set_index("uid").geometry.to_dict()

a_area_by_uid = gdf_a.set_index("uid")["area_m2"].to_dict()
b_area_by_uid = gdf_b.set_index("uid")["area_m2"].to_dict()

In [40]:
def safe_union(geoms):
    geoms = [g for g in geoms if g is not None and not g.is_empty]
    if not geoms:
        return None
    return unary_union(geoms)

In [41]:
def get_union_area(geoms):
    union_geom = safe_union(geoms)
    if union_geom is None:
        return np.nan
    return union_geom.area

In [42]:
components_df["sum_area_a"] = components_df["uids_a"].apply(
    lambda ids: sum(a_area_by_uid[x] for x in ids) if ids else 0.0
)

components_df["sum_area_b"] = components_df["uids_b"].apply(
    lambda ids: sum(b_area_by_uid[x] for x in ids) if ids else 0.0
)

components_df["union_area_a"] = components_df["uids_a"].apply(
    lambda ids: get_union_area([a_geom_by_uid[x] for x in ids]) if ids else np.nan
)

components_df["union_area_b"] = components_df["uids_b"].apply(
    lambda ids: get_union_area([b_geom_by_uid[x] for x in ids]) if ids else np.nan
)

components_df["union_area_all"] = components_df.apply(
    lambda row: get_union_area(
        [a_geom_by_uid[x] for x in row["uids_a"]] +
        [b_geom_by_uid[x] for x in row["uids_b"]]
    ),
    axis=1
)

In [43]:
print("Агрегированные геометрические признаки компонент добавлены.")
components_df.head()

Агрегированные геометрические признаки компонент добавлены.


,component_id,uids_a,uids_b,n_a,n_b,match_type,n_edges_ab,max_iou,mean_iou,max_overlap_a,max_overlap_b,min_dist_m,sum_area_a,sum_area_b,union_area_a,union_area_b,union_area_all
0,1,"[A_0, A_3927, A_4013, A_4044, A_4134, A_4167, ...",[B_5345],8,1,n:1,8,0.099540,0.058140,0.965276,0.099897,0.0,759.985933,1452.385070,759.985933,1452.385070,1532.167560
1,2,"[A_1, A_4185, A_4263, A_4352, A_4414]",[B_5585],5,1,n:1,5,0.150684,0.093211,0.987838,0.151449,0.0,697.799234,1397.525977,696.138900,1397.525977,1440.170759
2,3,"[A_19400, A_19424, A_19431, A_19470, A_19514, ...","[B_24877, B_24899, B_24937, B_25028, B_25044, ...",12,8,n:n,27,0.489267,0.096725,1.000000,1.000000,0.0,10015.155159,3172.843909,9499.593796,3172.843909,9609.025318
3,4,[A_3],[],1,0,A_only,0,NaN,NaN,NaN,NaN,NaN,64.058721,0.000000,64.058721,NaN,64.058721
4,5,[A_4],[],1,0,A_only,0,NaN,NaN,NaN,NaN,NaN,170.719519,0.000000,170.719519,NaN,170.719519


## Шаг 9 - Выбор репрезентативной геометрии для итоговой сущности

---

Теперь нужно решить, какую геометрию хранить как итоговую геометрию здания.

Мы не будем брать union A+B как основную геометрию по умолчанию, потому что объединение контуров из разных источников часто даёт рваный, синтетический и не очень интерпретируемый контур.

Вместо этого используем аккуратную эвристику:
1. Если в компоненте есть только A, то берём union геометрий A.
2. Если в компоненте есть только B, то берём union геометрий B.
3. Если есть и A, и B - сравниваем, какой источник даёт более компактную и цельную геометрию, а затем берём ту сторону, чья union-площадь ближе к общей объединённой площади компоненты и при этом не выглядит слишком раздробленной.

Используем правило
- если компонент типа 1:1, то берём геометрию B, так как в B есть высота;
- если есть B и B не сильно "проигрывает" по покрытию, то берём union B;
- иначе берём union A.

In [44]:
def pick_geometry_for_component(uids_a, uids_b, match_type):
    geom_a = safe_union([a_geom_by_uid[x] for x in uids_a]) if uids_a else None
    geom_b = safe_union([b_geom_by_uid[x] for x in uids_b]) if uids_b else None

    if geom_a is None and geom_b is None:
        return None, "none"

    if geom_a is not None and geom_b is None:
        return geom_a, "A"

    if geom_b is not None and geom_a is None:
        return geom_b, "B"

    # Если связь простая 1:1, в качестве репрезентативной геометрии
    # берём B как источник, содержащий целевую высоту.
    if match_type == "1:1":
        return geom_b, "B"

    area_a = geom_a.area if geom_a is not None else np.nan
    area_b = geom_b.area if geom_b is not None else np.nan

    # Простое правило:
    # если геометрии сопоставимы по площади, предпочитаем B;
    # если B сильно отличается, берём A как более "основной" контур.
    ratio = min(area_a, area_b) / max(area_a, area_b) if max(area_a, area_b) > 0 else 0

    if ratio >= 0.70:
        return geom_b, "B"
    else:
        return geom_a, "A"

In [45]:
picked = components_df.apply(
    lambda row: pick_geometry_for_component(row["uids_a"], row["uids_b"], row["match_type"]),
    axis=1
)

components_df["rep_geometry"] = [x[0] for x in picked]
components_df["geometry_source"] = [x[1] for x in picked]

components_gdf = gpd.GeoDataFrame(
    components_df.copy(),
    geometry="rep_geometry",
    crs=TARGET_CRS
)

In [46]:
print("Итоговая репрезентативная геометрия выбрана.")
print(components_gdf["geometry_source"].value_counts(dropna=False))
components_gdf.head()

Итоговая репрезентативная геометрия выбрана.
geometry_source
B    99332
A    40317
Name: count, dtype: int64


,component_id,uids_a,uids_b,n_a,n_b,match_type,n_edges_ab,max_iou,mean_iou,max_overlap_a,max_overlap_b,min_dist_m,sum_area_a,sum_area_b,union_area_a,union_area_b,union_area_all,rep_geometry,geometry_source
0,1,"[A_0, A_3927, A_4013, A_4044, A_4134, A_4167, ...",[B_5345],8,1,n:1,8,0.099540,0.058140,0.965276,0.099897,0.0,759.985933,1452.385070,759.985933,1452.385070,1532.167560,"MULTIPOLYGON (((673826.469 6635465.722, 673830...",A
1,2,"[A_1, A_4185, A_4263, A_4352, A_4414]",[B_5585],5,1,n:1,5,0.150684,0.093211,0.987838,0.151449,0.0,697.799234,1397.525977,696.138900,1397.525977,1440.170759,"MULTIPOLYGON (((673857.382 6635477.207, 673858...",A
2,3,"[A_19400, A_19424, A_19431, A_19470, A_19514, ...","[B_24877, B_24899, B_24937, B_25028, B_25044, ...",12,8,n:n,27,0.489267,0.096725,1.000000,1.000000,0.0,10015.155159,3172.843909,9499.593796,3172.843909,9609.025318,"POLYGON ((677035.477 6640339.744, 677030.486 6...",A
3,4,[A_3],[],1,0,A_only,0,NaN,NaN,NaN,NaN,NaN,64.058721,0.000000,64.058721,NaN,64.058721,"MULTIPOLYGON (((677472.729 6654073.108, 677483...",A
4,5,[A_4],[],1,0,A_only,0,NaN,NaN,NaN,NaN,NaN,170.719519,0.000000,170.719519,NaN,170.719519,"MULTIPOLYGON (((677468.825 6654072.92, 677468....",A


## Шаг 10 - Перенос и агрегация атрибутов из B на уровень компоненты

---

Для компонентов, содержащих объекты B, считаем:
- median_height_b;
- median_stairs_b;
- median_avg_floor_height_b;
- число объектов B, внесших вклад в агрегацию.

In [47]:
b_attr = gdf_b.set_index("uid")[["height", "stairs", "avg_floor_height", "purpose_of_building"]].copy()

In [48]:
def aggregate_b_attributes(uids_b):
    if not uids_b:
        return pd.Series({
            "n_b_with_height": 0,
            "median_height_b": np.nan,
            "median_stairs_b": np.nan,
            "median_avg_floor_height_b": np.nan,
            "mode_purpose_b": np.nan,
        })

    tmp = b_attr.loc[uids_b].copy()

    purpose_mode = tmp["purpose_of_building"].mode()
    purpose_value = purpose_mode.iloc[0] if len(purpose_mode) > 0 else np.nan

    return pd.Series({
        "n_b_with_height": tmp["height"].notna().sum(),
        "median_height_b": tmp["height"].median(),
        "median_stairs_b": tmp["stairs"].median(),
        "median_avg_floor_height_b": tmp["avg_floor_height"].median(),
        "mode_purpose_b": purpose_value,
    })

In [49]:
b_agg = components_gdf["uids_b"].apply(aggregate_b_attributes)
components_gdf = pd.concat([components_gdf, b_agg], axis=1)

print("Атрибуты B агрегированы на уровень компонент.")
components_gdf[
    ["match_type", "n_b_with_height", "median_height_b", "median_stairs_b", "mode_purpose_b"]
].head()

Атрибуты B агрегированы на уровень компонент.


,match_type,n_b_with_height,median_height_b,median_stairs_b,mode_purpose_b
0,n:1,1.0,4.5,1.0,Нежилое здание
1,n:1,1.0,4.5,1.0,Нежилое здание
2,n:n,8.0,60.0,20.0,Жилое здание
3,A_only,0.0,NaN,NaN,NaN
4,A_only,0.0,NaN,NaN,NaN


## Шаг 11 - Оценка уверенности сопоставления

---

Простая логика уверенности в сопоставлении:

**high:**
- сильное перекрытие и понятная межисточниковая связь

**medium:**
- связь есть, но перекрытие умеренное

**low:**
- компонент смешанный, но сильного подтверждения нет

**none:**
- компонент односторонний (A_only или B_only)

In [50]:
def assign_match_confidence(row):
    if row["match_type"] in {"A_only", "B_only"}:
        return "none"

    max_iou = row["max_iou"]
    max_oa = row["max_overlap_a"]
    max_ob = row["max_overlap_b"]

    if (
        pd.notna(max_iou) and max_iou >= 0.50
    ) or (
        pd.notna(max_oa) and max_oa >= 0.80
    ) or (
        pd.notna(max_ob) and max_ob >= 0.80
    ):
        return "high"

    if (
        pd.notna(max_iou) and max_iou >= 0.15
    ) or (
        pd.notna(max_oa) and max_oa >= 0.60
    ) or (
        pd.notna(max_ob) and max_ob >= 0.60
    ):
        return "medium"

    return "low"

In [51]:
components_gdf["match_confidence"] = components_gdf.apply(assign_match_confidence, axis=1)

print("Распределение уверенности:")
print(components_gdf["match_confidence"].value_counts(dropna=False))

Распределение уверенности:
match_confidence
none      66070
high      45757
medium    22673
low        5149
Name: count, dtype: int64


## Формирование целевой переменной и признаков источника высоты

---

В итоговом датасете важно явно разделить:
1. наблюдаемую высоту, которую можно использовать как target при обучении
2. источник этой высоты
3. факт наличия/отсутствия достоверного target

Логика здесь такая:
- в рамках предоставленных данных реальная высота приходит только из источника B
- на уровне компоненты графа мы уже агрегировали высоту B в median_height_b
- значит именно median_height_b становится observed target
- для A_only-компонент целевая переменная остаётся NaN, и именно для них затем должна работать модель восстановления высоты

In [52]:
components_gdf["target_height"] = components_gdf["median_height_b"]
components_gdf["target_height_is_observed"] = components_gdf["target_height"].notna().astype("int8")

def assign_height_source(row):
    if pd.notna(row["median_height_b"]):
        return "B_observed"
    return "missing"

def assign_height_source_detail(row):
    if pd.isna(row["median_height_b"]):
        return "missing"

    # Детализация полезна для последующего анализа качества обучающей выборки.
    if row["match_type"] == "B_only":
        return "B_only_direct"
    if row["match_type"] == "1:1":
        return "B_from_1to1_match"
    if row["match_type"] == "1:n":
        return "B_from_1ton_match"
    if row["match_type"] == "n:1":
        return "B_from_nto1_match"
    if row["match_type"] == "n:n":
        return "B_from_nton_match"
    return "B_observed_other"

components_gdf["target_height_source"] = components_gdf.apply(assign_height_source, axis=1)
components_gdf["target_height_source_detail"] = components_gdf.apply(assign_height_source_detail, axis=1)

# Дополнительный полезный флаг:
# можно ли считать observed target относительно надежным для обучения.
# Например, самые надежные наблюдения — B_only и 1:1/high.
def assign_target_reliability(row):
    if row["target_height_is_observed"] == 0:
        return "none"

    if row["match_type"] == "B_only":
        return "high"

    if row["match_type"] == "1:1" and row["match_confidence"] == "high":
        return "high"

    if row["match_confidence"] in {"high", "medium"}:
        return "medium"

    return "low"

components_gdf["target_height_reliability"] = components_gdf.apply(assign_target_reliability, axis=1)

print("Поля target/source добавлены.")
print(
    components_gdf[
        [
            "target_height",
            "target_height_is_observed",
            "target_height_source",
            "target_height_source_detail",
            "target_height_reliability",
        ]
    ].head()
)

Поля target/source добавлены.
   target_height  target_height_is_observed target_height_source  \
0            4.5                          1           B_observed   
1            4.5                          1           B_observed   
2           60.0                          1           B_observed   
3            NaN                          0              missing   
4            NaN                          0              missing   

  target_height_source_detail target_height_reliability  
0           B_from_nto1_match                    medium  
1           B_from_nto1_match                    medium  
2           B_from_nton_match                    medium  
3                     missing                      none  
4                     missing                      none  


## Признаки соседнего окружения

---

Для восстановления пропущенной высоты полезно описать локальный контекст - сколько рядом зданий и каковы их высоты.

In [53]:
components_gdf = components_gdf.set_geometry("rep_geometry").copy()

# Центроиды храним отдельно как служебную геометрию для поиска соседей
components_centroids = components_gdf.copy()
components_centroids["geometry"] = components_centroids.geometry.centroid

# Пространственный индекс
sindex = components_centroids.sindex

In [54]:
def compute_neighbor_features(gdf_centroids, radii=(50, 100)):
    """
    Считает признаки локального окружения для каждой компоненты:
    - число соседей в радиусе;
    - число соседей с известной высотой;
    - статистики высот соседей.
    """
    rows = []

    geom_arr = gdf_centroids.geometry.values
    height_arr = gdf_centroids["target_height"].values
    comp_id_arr = gdf_centroids["component_id"].values

    for i, geom in enumerate(geom_arr):
        row = {"component_id": comp_id_arr[i]}

        for r in radii:
            # Кандидаты через spatial index по bbox буфера
            bbox = geom.buffer(r).bounds
            candidate_idx = list(sindex.intersection(bbox))

            # Убираем сам объект
            candidate_idx = [j for j in candidate_idx if j != i]

            if not candidate_idx:
                row[f"n_neighbors_{r}m"] = 0
                row[f"n_neighbors_obs_height_{r}m"] = 0
                row[f"neighbor_height_mean_{r}m"] = np.nan
                row[f"neighbor_height_median_{r}m"] = np.nan
                row[f"neighbor_height_min_{r}m"] = np.nan
                row[f"neighbor_height_max_{r}m"] = np.nan
                row[f"neighbor_height_std_{r}m"] = np.nan
                row[f"neighbor_height_q25_{r}m"] = np.nan
                row[f"neighbor_height_q75_{r}m"] = np.nan
                continue

            # Точное расстояние
            dists = gdf_centroids.geometry.iloc[candidate_idx].distance(geom)
            near_idx = [candidate_idx[k] for k, d in enumerate(dists) if d <= r]

            row[f"n_neighbors_{r}m"] = len(near_idx)

            if len(near_idx) == 0:
                row[f"n_neighbors_obs_height_{r}m"] = 0
                row[f"neighbor_height_mean_{r}m"] = np.nan
                row[f"neighbor_height_median_{r}m"] = np.nan
                row[f"neighbor_height_min_{r}m"] = np.nan
                row[f"neighbor_height_max_{r}m"] = np.nan
                row[f"neighbor_height_std_{r}m"] = np.nan
                row[f"neighbor_height_q25_{r}m"] = np.nan
                row[f"neighbor_height_q75_{r}m"] = np.nan
                continue

            neighbor_heights = pd.Series(height_arr[near_idx]).dropna()

            row[f"n_neighbors_obs_height_{r}m"] = len(neighbor_heights)

            if len(neighbor_heights) == 0:
                row[f"neighbor_height_mean_{r}m"] = np.nan
                row[f"neighbor_height_median_{r}m"] = np.nan
                row[f"neighbor_height_min_{r}m"] = np.nan
                row[f"neighbor_height_max_{r}m"] = np.nan
                row[f"neighbor_height_std_{r}m"] = np.nan
                row[f"neighbor_height_q25_{r}m"] = np.nan
                row[f"neighbor_height_q75_{r}m"] = np.nan
            else:
                row[f"neighbor_height_mean_{r}m"] = neighbor_heights.mean()
                row[f"neighbor_height_median_{r}m"] = neighbor_heights.median()
                row[f"neighbor_height_min_{r}m"] = neighbor_heights.min()
                row[f"neighbor_height_max_{r}m"] = neighbor_heights.max()
                row[f"neighbor_height_std_{r}m"] = neighbor_heights.std()
                row[f"neighbor_height_q25_{r}m"] = neighbor_heights.quantile(0.25)
                row[f"neighbor_height_q75_{r}m"] = neighbor_heights.quantile(0.75)

        rows.append(row)

    return pd.DataFrame(rows)

neighbor_features = compute_neighbor_features(components_centroids, radii=(50, 100))

components_gdf = components_gdf.merge(
    neighbor_features,
    on="component_id",
    how="left"
)

In [55]:
print("Соседские признаки добавлены.")
print(
    components_gdf[
        [
            "component_id",
            "n_neighbors_50m",
            "n_neighbors_obs_height_50m",
            "neighbor_height_median_50m",
            "n_neighbors_100m",
            "n_neighbors_obs_height_100m",
            "neighbor_height_median_100m",
        ]
    ].head()
)

Соседские признаки добавлены.
   component_id  n_neighbors_50m  n_neighbors_obs_height_50m  \
0             1               17                          16   
1             2               13                          13   
2             3                2                           2   
3             4                5                           2   
4             5                7                           4   

   neighbor_height_median_50m  n_neighbors_100m  n_neighbors_obs_height_100m  \
0                        4.50                51                           50   
1                        4.50                36                           35   
2                       32.25                 9                            7   
3                        4.50                13                            9   
4                        4.50                13                            9   

   neighbor_height_median_100m  
0                          4.5  
1                          4.5  
2    

## Шаг 12 - сохранение

На этом шаге делаем итоговую таблицу. Здесь каждая строка = одна компонента графа = один кандидат на физическое здание / группу секций одного здания.

Сохраняем:
- состав компоненты
- тип мэтча
- источник выбранной геометрии
- базовые агрегированные атрибуты
- confidence merge
- итоговую geometry

In [56]:
merged_buildings = components_gdf.copy()

# Упорядочим основные поля
main_cols = [
    "component_id",
    "match_type",
    "match_confidence",
    "geometry_source",

    "target_height",
    "target_height_is_observed",
    "target_height_source",
    "target_height_source_detail",
    "target_height_reliability",

    "n_a",
    "n_b",
    "uids_a",
    "uids_b",

    "n_edges_ab",
    "max_iou",
    "mean_iou",
    "max_overlap_a",
    "max_overlap_b",
    "min_dist_m",

    "sum_area_a",
    "sum_area_b",
    "union_area_a",
    "union_area_b",
    "union_area_all",

    "n_b_with_height",
    "median_height_b",
    "median_stairs_b",
    "median_avg_floor_height_b",
    "mode_purpose_b",

    "n_neighbors_50m",
    "n_neighbors_obs_height_50m",
    "neighbor_height_mean_50m",
    "neighbor_height_median_50m",
    "neighbor_height_min_50m",
    "neighbor_height_max_50m",
    "neighbor_height_std_50m",
    "neighbor_height_q25_50m",
    "neighbor_height_q75_50m",

    "n_neighbors_100m",
    "n_neighbors_obs_height_100m",
    "neighbor_height_mean_100m",
    "neighbor_height_median_100m",
    "neighbor_height_min_100m",
    "neighbor_height_max_100m",
    "neighbor_height_std_100m",
    "neighbor_height_q25_100m",
    "neighbor_height_q75_100m",

    "rep_geometry",
]

In [57]:
other_cols = [c for c in merged_buildings.columns if c not in main_cols]
merged_buildings = merged_buildings[main_cols + other_cols]

In [58]:
print("Итоговый датасет оформлен.")
print(merged_buildings.shape)
merged_buildings.head()

Итоговый датасет оформлен.
(139649, 48)


,component_id,match_type,match_confidence,geometry_source,target_height,target_height_is_observed,target_height_source,target_height_source_detail,target_height_reliability,n_a,...,n_neighbors_100m,n_neighbors_obs_height_100m,neighbor_height_mean_100m,neighbor_height_median_100m,neighbor_height_min_100m,neighbor_height_max_100m,neighbor_height_std_100m,neighbor_height_q25_100m,neighbor_height_q75_100m,rep_geometry
0,1,n:1,high,A,4.5,1,B_observed,B_from_nto1_match,medium,8,...,51,50,5.256000,4.5,4.5,6.60,1.018234,4.5,6.6,"MULTIPOLYGON (((673826.469 6635465.722, 673830..."
1,2,n:1,high,A,4.5,1,B_observed,B_from_nto1_match,medium,5,...,36,35,4.980000,4.5,4.5,6.60,0.894690,4.5,4.5,"MULTIPOLYGON (((673857.382 6635477.207, 673858..."
2,3,n:n,high,A,60.0,1,B_observed,B_from_nton_match,medium,12,...,9,7,26.321429,18.0,4.5,74.75,29.001693,4.5,39.0,"POLYGON ((677035.477 6640339.744, 677030.486 6..."
3,4,A_only,none,A,NaN,0,missing,missing,none,1,...,13,9,9.333333,4.5,4.5,48.00,14.500000,4.5,4.5,"MULTIPOLYGON (((677472.729 6654073.108, 677483..."
4,5,A_only,none,A,NaN,0,missing,missing,none,1,...,13,9,9.333333,4.5,4.5,48.00,14.500000,4.5,4.5,"MULTIPOLYGON (((677468.825 6654072.92, 677468...."


In [59]:
total_components = len(merged_buildings)
matched_components = (~merged_buildings["match_type"].isin(["A_only", "B_only"])).sum()
a_only_components = (merged_buildings["match_type"] == "A_only").sum()
b_only_components = (merged_buildings["match_type"] == "B_only").sum()
with_height_from_b = merged_buildings["median_height_b"].notna().sum()

print("=" * 70)
print("СВОДКА ПО РЕЗУЛЬТАТАМ MERGE")
print("=" * 70)

print(f"Всего компонент: {total_components:,}")
print(f"Смешанных (matched) компонент: {matched_components:,}")
print(f"Только A: {a_only_components:,}")
print(f"Только B: {b_only_components:,}")
print(f"Компонент с наблюдаемой высотой из B: {with_height_from_b:,}")

print("\nТипы матчей:")
print(merged_buildings["match_type"].value_counts(dropna=False))

print("\nConfidence:")
print(merged_buildings["match_confidence"].value_counts(dropna=False))

print("\nКомпоненты с высотой из B по типам матчей:")
print(
    merged_buildings.assign(has_height=merged_buildings["median_height_b"].notna())
    .groupby("match_type")["has_height"]
    .mean()
    .sort_values(ascending=False)
)

СВОДКА ПО РЕЗУЛЬТАТАМ MERGE
Всего компонент: 139,649
Смешанных (matched) компонент: 73,579
Только A: 38,086
Только B: 27,984
Компонент с наблюдаемой высотой из B: 101,563

Типы матчей:
match_type
1:1       49170
A_only    38086
B_only    27984
n:n       12201
1:n        6347
n:1        5861
Name: count, dtype: int64

Confidence:
match_confidence
none      66070
high      45757
medium    22673
low        5149
Name: count, dtype: int64

Компоненты с высотой из B по типам матчей:
match_type
1:1       1.0
1:n       1.0
B_only    1.0
n:1       1.0
n:n       1.0
A_only    0.0
Name: has_height, dtype: float64


In [60]:
MERGED_PARQUET = OUTPUT_DIR / "merged_buildings_by_geometry.parquet"
MERGED_GPKG = OUTPUT_DIR / "merged_buildings_by_geometry.gpkg"
MERGED_CSV = OUTPUT_DIR / "merged_buildings_by_geometry.csv"

EDGES_CSV = OUTPUT_DIR / "merge_edges_ab.csv"
CANDIDATES_CSV = OUTPUT_DIR / "merge_candidate_pairs_metrics.csv"

# Итоговые merged-сущности
merged_buildings.to_parquet(MERGED_PARQUET)
merged_buildings.to_file(MERGED_GPKG, driver="GPKG")

merged_buildings.drop(columns=["rep_geometry"]).to_csv(MERGED_CSV, index=False)

# Таблицы связей и кандидатов
edges_ab.to_csv(EDGES_CSV, index=False)
candidate_pairs.to_csv(CANDIDATES_CSV, index=False)

print("Файлы сохранены:")
print(MERGED_PARQUET)
print(MERGED_GPKG)
print(MERGED_CSV)
print(EDGES_CSV)
print(CANDIDATES_CSV)

Файлы сохранены:
..\data\interim\merged_buildings_by_geometry.parquet
..\data\interim\merged_buildings_by_geometry.gpkg
..\data\interim\merged_buildings_by_geometry.csv
..\data\interim\merge_edges_ab.csv
..\data\interim\merge_candidate_pairs_metrics.csv
